# Official fast.ai course22 original 05 — Colab adaptation

Original: [05-linear-model-and-neural-net-from-scratch.ipynb](https://github.com/fastai/course22/blob/230390584ba5b990e20487d68681a9ec5b258eff/clean/05-linear-model-and-neural-net-from-scratch.ipynb)

Pinned source commit: `230390584ba5b990e20487d68681a9ec5b258eff`. Source SHA256: `40f30499aa4c3a0cb73f621c0f2a810b375d8cf70e510de5b28030d9fdde1092`.

This additional notebook preserves the full original computational experiments. Existing independently written companions 00–07 remain separate. Set QUICK_RUN=False; run every code cell. Source notebooks and authorship belong to Jeremy Howard and fast.ai. No separate course22 repository license was found at the pinned commit; no fastbook license is applied to these files.

- Original Kaggle train/test data sourced from pinned public teaching copies. All891/418 passengers, column names, values and Name/Ticket/Cabin features verified against a second independent competition mirror; training values also equal the DataScienceDojo copy. No Kaggle provider checksum/terms acceptance/account score is claimed. Test labels remain unavailable.
- Pandas2 bool dummy columns converted explicitly to float32 before constructing tensors. Corrected argument order in sigmoid test prediction. Original full feature normalization before the split is retained as an identified teaching limitation. All six training experiments preserve epochs, learning rates, MAE objective and seed442; checkpoints/validation evidence added for every epoch/experiment.

Public publication and competition submission have not been performed. Saved predictions are learning artifacts; the 418 Titanic test targets are unavailable. Original conceptual prompts/headings are retained below.


In [ ]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '03_fastai', 'fastai', '22_original_05_titanic_from_scratch.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)
if IN_COLAB:
    from packaging.requirements import Requirement
    import importlib.metadata as _metadata, subprocess
    _missing = []
    for _spec in ['sympy>=1.13,<2']:
        _requirement = Requirement(_spec)
        try: _version = _metadata.version(_requirement.name)
        except _metadata.PackageNotFoundError: _version = None
        if _version is None or _version not in _requirement.specifier: _missing.append(_spec)
    if _missing: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *_missing])

In [ ]:
# Full-source execution and durable evidence helpers (Colab adaptation).
import hashlib, json, shutil, urllib.request
from datetime import datetime, timezone
if QUICK_RUN or os.environ.get('COURSE_SMOKE') == '1':
    raise ValueError('These original notebooks require full mode: set QUICK_RUN=False.')
RESULTS = COLAB_CONTEXT['folder'] / 'results' / Path(COLAB_CONTEXT['run']['notebook']).stem
MODELS = COLAB_CONTEXT['folder'] / 'models' / Path(COLAB_CONTEXT['run']['notebook']).stem
WORK = COLAB_CONTEXT['folder'] / 'data' / Path(COLAB_CONTEXT['run']['notebook']).stem
for directory in [RESULTS, MODELS, WORK]: directory.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

def record_stage(stage, **evidence):
    target = RESULTS / 'stage_evidence.json'
    started = COLAB_CONTEXT['run']['started_utc']
    data = json.loads(target.read_text(encoding='utf-8')) if target.is_file() else {}
    if data.get('run_started_utc') != started:
        data = {'notebook': COLAB_CONTEXT['run']['notebook'], 'run_started_utc': started, 'stages': {}}
    data['stages'][stage] = {'recorded_utc': datetime.now(timezone.utc).isoformat(), **evidence}
    data.update(quick_run=False, status='observed_stage_outputs', provider_course_completion_claimed=False,
                learner_personal_reflection_complete=False)
    temporary = target.with_suffix('.writing')
    temporary.write_text(json.dumps(data, indent=2, default=str), encoding='utf-8')
    temporary.replace(target)

def verified_download(url, target, sha256):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.is_file():
        temporary = target.with_name(target.name + '.download')
        with urllib.request.urlopen(url, timeout=120) as response, temporary.open('wb') as out:
            shutil.copyfileobj(response, out)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != sha256:
            raise ValueError('Source download checksum mismatch: ' + url)
        temporary.replace(target)
    if hashlib.sha256(target.read_bytes()).hexdigest() != sha256:
        raise ValueError('Existing source file differs from the pinned original: ' + str(target))
    return target

def save_plot(figure, name):
    if hasattr(figure, 'write_html'):
        figure.write_html(RESULTS / (name + '.html'), include_plotlyjs=True)
    elif hasattr(figure, 'savefig'):
        figure.savefig(RESULTS / (name + '.png'), dpi=150, bbox_inches='tight')
    return figure

record_stage('full_mode_setup', execution_location='hosted_colab' if IN_COLAB else 'local_validation',
             original_exercises_verified_complete=False)

## Evidence required for the original exercises

- Execute every nonempty original code cell top to bottom in hosted Colab; retain outputs and execution records.
- QUICK_RUN=False. Original experiment data coverage, epochs and learning rates are preserved.
- Original source attribution and the per-cell provenance map distinguish original exercises from companions.
- Save full notebook outputs and actual stage/checkpoint/artifact evidence in Drive.
- No provider badge, external publishing, competition submission or personal authorship is certified by execution.
- Use the full original891 training/418 unlabeled test passengers and all12 source features. Preserve raw linear18 epochs lr.2; sigmoid30 lr100; matrix30 lr100; hidden20 networks30 lr1.4 and30 lr20; deep[10,10]30 lr4; all gradients, preprocessing, validation and test CSV experiments.


## Introduction

In [ ]:
# Exact public teaching copies of the original Kaggle competition files.
# No credentials, test labels, rule acceptance, or competition submission are inferred.
path = COLAB_CONTEXT['folder'] / 'data' / 'titanic_original'
_titanic_sources = [{'file': 'train.csv', 'url': 'https://raw.githubusercontent.com/minsuk-heo/kaggle-titanic/96c14e776090f7919d15064ed73e3b2788b4288c/input/train.csv', 'sha256': '7d118fef8b6ccf7f81111877bc388536f7b1e498a655e3d649d19aaa010e9f6f', 'rows': 891, 'columns': ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'], 'all_values_equal_independent_dsindy_pin': True, 'all_values_equal_datadojo': True}, {'file': 'test.csv', 'url': 'https://raw.githubusercontent.com/minsuk-heo/kaggle-titanic/96c14e776090f7919d15064ed73e3b2788b4288c/input/test.csv', 'sha256': '56023b9948236f3c7a1c9448fcf418b283e109ef177fa8c7e069158dd7dd52b2', 'rows': 418, 'columns': ['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'], 'all_values_equal_independent_dsindy_pin': True}]
for _source in _titanic_sources:
    verified_download(_source['url'], path / _source['file'], _source['sha256'])
import pandas as pd
_train = pd.read_csv(path / 'train.csv')
_test = pd.read_csv(path / 'test.csv')
assert _train.shape == (891, 12) and _test.shape == (418, 11)
assert _train.PassengerId.tolist() == list(range(1, 892))
assert _test.PassengerId.tolist() == list(range(892, 1310))
assert 'Survived' not in _test and int(_train.Survived.sum()) == 342
record_stage('original_titanic_inputs', sources=_titanic_sources, train_rows=891, test_rows=418,
             held_out_test_labels_available=False, competition_submission_performed=False)

In [ ]:
import torch, numpy as np, pandas as pd
np.set_printoptions(linewidth=140)
torch.set_printoptions(linewidth=140, sci_mode=False, edgeitems=7)
pd.set_option('display.width', 140)

## Cleaning the data

In [ ]:
df = pd.read_csv(path/'train.csv')
df

In [ ]:
df.isna().sum()

In [ ]:
modes = df.mode().iloc[0]
modes

In [ ]:
df.fillna(modes, inplace=True)

In [ ]:
df.isna().sum()

In [ ]:
import numpy as np

df.describe(include=(np.number))

In [ ]:
df['Fare'].hist();

In [ ]:
df['LogFare'] = np.log(df['Fare']+1)

In [ ]:
df['LogFare'].hist();

In [ ]:
pclasses = sorted(df.Pclass.unique())
pclasses

In [ ]:
df.describe(include=[object])

In [ ]:
df = pd.get_dummies(df, columns=["Sex","Pclass","Embarked"])
df.columns

In [ ]:
added_cols = ['Sex_male', 'Sex_female', 'Pclass_1', 'Pclass_2', 'Pclass_3', 'Embarked_C', 'Embarked_Q', 'Embarked_S']
df[added_cols].head()

In [ ]:
from torch import tensor

t_dep = tensor(df.Survived)

In [ ]:
indep_cols = ['Age', 'SibSp', 'Parch', 'LogFare'] + added_cols

t_indep = tensor(df[indep_cols].to_numpy(dtype=np.float32), dtype=torch.float)
t_indep

In [ ]:
t_indep.shape

## Setting up a linear model

In [ ]:
torch.manual_seed(442)

n_coeff = t_indep.shape[1]
coeffs = torch.rand(n_coeff)-0.5
coeffs

In [ ]:
t_indep*coeffs

In [ ]:
vals,indices = t_indep.max(dim=0)
t_indep = t_indep / vals

In [ ]:
t_indep*coeffs

In [ ]:
preds = (t_indep*coeffs).sum(axis=1)

In [ ]:
preds[:10]

In [ ]:
loss = torch.abs(preds-t_dep).mean()
loss

In [ ]:
def calc_preds(coeffs, indeps): return (indeps*coeffs).sum(axis=1)
def calc_loss(coeffs, indeps, deps): return torch.abs(calc_preds(coeffs, indeps)-deps).mean()

## Doing a gradient descent step

In [ ]:
coeffs.requires_grad_()

In [ ]:
loss = calc_loss(coeffs, t_indep, t_dep)
loss

In [ ]:
loss.backward()

In [ ]:
coeffs.grad

In [ ]:
loss = calc_loss(coeffs, t_indep, t_dep)
loss.backward()
coeffs.grad

In [ ]:
loss = calc_loss(coeffs, t_indep, t_dep)
loss.backward()
with torch.no_grad():
    coeffs.sub_(coeffs.grad * 0.1)
    coeffs.grad.zero_()
    print(calc_loss(coeffs, t_indep, t_dep))

## Training the linear model

In [ ]:
from fastai.data.transforms import RandomSplitter
trn_split,val_split=RandomSplitter(seed=42)(df)

In [ ]:
trn_indep,val_indep = t_indep[trn_split],t_indep[val_split]
trn_dep,val_dep = t_dep[trn_split],t_dep[val_split]
len(trn_indep),len(val_indep)

In [ ]:
def update_coeffs(coeffs, lr):
    coeffs.sub_(coeffs.grad * lr)
    coeffs.grad.zero_()

In [ ]:
def one_epoch(coeffs, lr):
    loss = calc_loss(coeffs, trn_indep, trn_dep)
    loss.backward()
    with torch.no_grad(): update_coeffs(coeffs, lr)
    print(f"{loss:.3f}", end="; ")
    return float(loss.detach())

In [ ]:
def init_coeffs(): return (torch.rand(n_coeff)-0.5).requires_grad_()

In [ ]:
_manual_experiment_index = 0
def train_model(epochs=30, lr=0.01):
    # Original schedule and initialization, with extra per-epoch Drive evidence.
    global _manual_experiment_index
    _manual_experiment_index += 1
    torch.manual_seed(442)
    coeffs = init_coeffs()
    history = []
    for i in range(epochs):
        history.append(one_epoch(coeffs, lr=lr))
        state_path = MODELS / f'manual_experiment_{_manual_experiment_index:02d}.pt'
        torch.save({'coeffs': coeffs, 'completed_epochs': i+1, 'requested_epochs': epochs,
                    'lr': lr, 'seed': 442, 'loss': 'mean_absolute_error', 'history': history}, state_path)
        record_stage(f'manual_experiment_{_manual_experiment_index:02d}', completed_epochs=i+1,
                     requested_epochs=epochs, learning_rate=lr, seed=442, loss='mean_absolute_error',
                     loss_history=history, checkpoint=str(state_path))
    with torch.no_grad():
        vp = calc_preds(coeffs, val_indep).reshape(-1)
        truth = val_dep.reshape(-1)
        assert vp.shape == truth.shape
        pd.DataFrame({'validation_position': list(val_split), 'target': truth.numpy(),
                      'prediction': vp.numpy()}).to_csv(RESULTS / f'manual_experiment_{_manual_experiment_index:02d}_validation.csv', index=False)
        record_stage(f'manual_experiment_{_manual_experiment_index:02d}_validation',
                     rows=len(truth), accuracy=float((truth.bool() == (vp > .5)).float().mean()),
                     mean_absolute_error=float((vp-truth).abs().mean()))
    return coeffs

In [ ]:
coeffs = train_model(18, lr=0.2)

In [ ]:
def show_coeffs(): return dict(zip(indep_cols, coeffs.requires_grad_(False)))
show_coeffs()

## Measuring accuracy

In [ ]:
preds = calc_preds(coeffs, val_indep)

In [ ]:
results = val_dep.bool()==(preds>0.5)
results[:16]

In [ ]:
results.float().mean()

In [ ]:
def acc(coeffs): return (val_dep.bool()==(calc_preds(coeffs, val_indep)>0.5)).float().mean()
acc(coeffs)

## Using sigmoid

In [ ]:
preds[:28]

In [ ]:
import sympy
_sigmoid_x = sympy.Symbol('x')
sympy.plot(1/(1+sympy.exp(-_sigmoid_x)), (_sigmoid_x, -5, 5), xlim=(-5,5));

In [ ]:
def calc_preds(coeffs, indeps): return torch.sigmoid((indeps*coeffs).sum(axis=1))

In [ ]:
coeffs = train_model(lr=100)

In [ ]:
acc(coeffs)

In [ ]:
show_coeffs()

## Submitting to Kaggle

In [ ]:
tst_df = pd.read_csv(path/'test.csv')

In [ ]:
tst_df['Fare'] = tst_df.Fare.fillna(0)

In [ ]:
tst_df.fillna(modes, inplace=True)
tst_df['LogFare'] = np.log(tst_df['Fare']+1)
tst_df = pd.get_dummies(tst_df, columns=["Sex","Pclass","Embarked"])

tst_indep = tensor(tst_df[indep_cols].to_numpy(dtype=np.float32), dtype=torch.float)
tst_indep = tst_indep / vals

In [ ]:
tst_df['Survived'] = (calc_preds(coeffs, tst_indep)>0.5).int()

In [ ]:
sub_df = tst_df[['PassengerId','Survived']]
sub_df.to_csv(RESULTS / 'sub.csv', index=False)
record_stage('sigmoid_test_predictions', rows=len(sub_df), csv=str(RESULTS / 'sub.csv'), test_labels_available=False, submitted=False)

In [ ]:
pd.read_csv(RESULTS / 'sub.csv').head()

## Using matrix product

In [ ]:
(val_indep*coeffs).sum(axis=1)

In [ ]:
val_indep@coeffs

In [ ]:
def calc_preds(coeffs, indeps): return torch.sigmoid(indeps@coeffs)

In [ ]:
def init_coeffs(): return (torch.rand(n_coeff, 1)*0.1).requires_grad_()

In [ ]:
trn_dep = trn_dep[:,None]
val_dep = val_dep[:,None]

In [ ]:
coeffs = train_model(lr=100)

In [ ]:
acc(coeffs)

## A neural network

In [ ]:
def init_coeffs(n_hidden=20):
    layer1 = (torch.rand(n_coeff, n_hidden)-0.5)/n_hidden
    layer2 = torch.rand(n_hidden, 1)-0.3
    const = torch.rand(1)[0]
    return layer1.requires_grad_(),layer2.requires_grad_(),const.requires_grad_()

In [ ]:
import torch.nn.functional as F

def calc_preds(coeffs, indeps):
    l1,l2,const = coeffs
    res = F.relu(indeps@l1)
    res = res@l2 + const
    return torch.sigmoid(res)

In [ ]:
def update_coeffs(coeffs, lr):
    for layer in coeffs:
        layer.sub_(layer.grad * lr)
        layer.grad.zero_()

In [ ]:
coeffs = train_model(lr=1.4)

In [ ]:
coeffs = train_model(lr=20)

In [ ]:
acc(coeffs)

## Deep learning

In [ ]:
def init_coeffs():
    hiddens = [10, 10]  # <-- set this to the size of each hidden layer you want
    sizes = [n_coeff] + hiddens + [1]
    n = len(sizes)
    layers = [(torch.rand(sizes[i], sizes[i+1])-0.3)/sizes[i+1]*4 for i in range(n-1)]
    consts = [(torch.rand(1)[0]-0.5)*0.1 for i in range(n-1)]
    for l in layers+consts: l.requires_grad_()
    return layers,consts

In [ ]:
import torch.nn.functional as F

def calc_preds(coeffs, indeps):
    layers,consts = coeffs
    n = len(layers)
    res = indeps
    for i,l in enumerate(layers):
        res = res@l + consts[i]
        if i!=n-1: res = F.relu(res)
    return torch.sigmoid(res)

In [ ]:
def update_coeffs(coeffs, lr):
    layers,consts = coeffs
    for layer in layers+consts:
        layer.sub_(layer.grad * lr)
        layer.grad.zero_()

In [ ]:
coeffs = train_model(lr=4)

In [ ]:
acc(coeffs)

## Final thoughts

In [ ]:
record_stage('reached_final_cell', all_previous_cells_independently_verified=False)
finish_run(COLAB_CONTEXT)